In [1]:
!pip install -q transformers accelerate safetensors scikit-learn

In [2]:
import torch
import numpy as np
import pandas as pd
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Verify the folder is visible
!ls /content/drive/MyDrive/hinemo

Mounted at /content/drive
checkpoints  DATASET_FOR_HINEMO  splits


In [4]:
# ── CHANGE ONLY THESE TWO LINES per Colab account ──────────────
HF_MODEL_ID  = "bert-base-multilingual-cased"  # HuggingFace model ID
MODEL_FOLDER = "bert-base-multilingual-cased"  # folder name inside checkpoints/
# ────────────────────────────────────────────────────────────────

# For MuRIL:
# HF_MODEL_ID  = "google/muril-base-cased"
# MODEL_FOLDER = "muril-base-cased"

# For XLM-R:
# HF_MODEL_ID  = "xlm-roberta-base"
# MODEL_FOLDER = "xlm-roberta-base"

# Paths — all derived from the two lines above
BASE_DIR       = "/content/drive/MyDrive/hinemo"
VAL_PATH       = f"{BASE_DIR}/splits/hinemo_val.csv"
CHECKPOINT_DIR = f"{BASE_DIR}/checkpoints/{MODEL_FOLDER}/best_model"
OUTPUT_DIR     = f"{BASE_DIR}/hidden_states/{MODEL_FOLDER}"

# Constants — never change these
BATCH_SIZE        = 32
MAX_LENGTH        = 128
RANDOM_SEED       = 524
SAMPLES_PER_CLASS = 750   # 750 × 4 emotions = 3,000 total

print(f"Model:       {HF_MODEL_ID}")
print(f"Checkpoint:  {CHECKPOINT_DIR}")
print(f"Output:      {OUTPUT_DIR}")

Model:       bert-base-multilingual-cased
Checkpoint:  /content/drive/MyDrive/hinemo/checkpoints/bert-base-multilingual-cased/best_model
Output:      /content/drive/MyDrive/hinemo/hidden_states/bert-base-multilingual-cased


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

Device: cuda
GPU: Tesla T4


In [6]:
# Load full val set
val_df = pd.read_csv(VAL_PATH)
print(f"Full val set: {len(val_df)} rows")
print(val_df["gpt_emotion"].value_counts())

# Stratified subsample: 750 per class
sample_df = (
    val_df
    .groupby("gpt_emotion", group_keys=False)
    .apply(lambda x: x.sample(SAMPLES_PER_CLASS, random_state=RANDOM_SEED))
    .reset_index(drop=True)
)

print(f"\nSubsample: {len(sample_df)} rows")
print(sample_df["gpt_emotion"].value_counts())

# Create output directory and save metadata
os.makedirs(OUTPUT_DIR, exist_ok=True)

metadata = sample_df[["id", "gpt_emotion", "lambda"]].reset_index(drop=True)
metadata.to_csv(f"{OUTPUT_DIR}/metadata.csv", index=False)
print(f"\nMetadata saved to {OUTPUT_DIR}/metadata.csv")

Full val set: 6926 rows
gpt_emotion
joy        2658
anger      1855
disgust    1304
sadness    1109
Name: count, dtype: int64

Subsample: 3000 rows
gpt_emotion
anger      750
disgust    750
joy        750
sadness    750
Name: count, dtype: int64

Metadata saved to /content/drive/MyDrive/hinemo/hidden_states/bert-base-multilingual-cased/metadata.csv


/tmp/ipykernel_498/1244759081.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(SAMPLES_PER_CLASS, random_state=RANDOM_SEED))


In [7]:
def extract_hidden_states(model_path, sample_df, tokenizer, label):
    print(f"\nLoading model from: {model_path}")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        output_hidden_states=True,
        ignore_mismatched_sizes=True   # suppresses warning for pretrained (no classification head yet)
    ).to(device).eval()

    all_hidden = []

    for i in tqdm(range(0, len(sample_df), BATCH_SIZE), desc=f"Extracting [{label}]"):
        batch_texts = sample_df["text"].iloc[i : i + BATCH_SIZE].tolist()

        encoded = tokenizer(
            batch_texts,
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoded)

        # hidden_states: tuple of 13 tensors, each (batch, seq_len, 768)
        # Skip index 0 (embedding layer), take layers 1-12
        # [:, 0, :] = CLS token only
        cls_per_layer = torch.stack(
            [h[:, 0, :] for h in outputs.hidden_states[1:]],
            dim=1
        )  # shape: (batch_size, 12, 768)

        all_hidden.append(cls_per_layer.cpu().numpy())

    # Stack all batches → (3000, 12, 768)
    hidden_array = np.concatenate(all_hidden, axis=0)

    # Save to Drive
    save_path = f"{OUTPUT_DIR}/hidden_{label}.npy"
    np.save(save_path, hidden_array)
    print(f"Saved: {save_path}  |  Shape: {hidden_array.shape}")

    # Free GPU memory before next run
    del model
    torch.cuda.empty_cache()

    return hidden_array

In [8]:
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
print(f"Tokenizer loaded: {HF_MODEL_ID}")

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Tokenizer loaded: bert-base-multilingual-cased


In [9]:
hidden_pretrained = extract_hidden_states(
    model_path = HF_MODEL_ID,
    sample_df  = sample_df,
    tokenizer  = tokenizer,
    label      = "pretrained"
)


Loading model from: bert-base-multilingual-cased


model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Extracting 

Saved: /content/drive/MyDrive/hinemo/hidden_states/bert-base-multilingual-cased/hidden_pretrained.npy  |  Shape: (3000, 12, 768)


In [10]:
hidden_finetuned = extract_hidden_states(
    model_path = CHECKPOINT_DIR,
    sample_df  = sample_df,
    tokenizer  = tokenizer,
    label      = "finetuned"
)


Loading model from: /content/drive/MyDrive/hinemo/checkpoints/bert-base-multilingual-cased/best_model


Loading weights:   0%|          | 0/201 [00:02<?, ?it/s]

Extracting [finetuned]: 100%|██████████| 94/94 [00:19<00:00,  4.93it/s]


Saved: /content/drive/MyDrive/hinemo/hidden_states/bert-base-multilingual-cased/hidden_finetuned.npy  |  Shape: (3000, 12, 768)


In [11]:
# Reload from Drive and verify shapes
h_pre  = np.load(f"{OUTPUT_DIR}/hidden_pretrained.npy")
h_fine = np.load(f"{OUTPUT_DIR}/hidden_finetuned.npy")
meta   = pd.read_csv(f"{OUTPUT_DIR}/metadata.csv")

print(f"Pretrained shape:  {h_pre.shape}   ← should be (3000, 12, 768)")
print(f"Finetuned shape:   {h_fine.shape}  ← should be (3000, 12, 768)")
print(f"Metadata rows:     {len(meta)}      ← should be 3000")
print(f"\nEmotion distribution in subsample:")
print(meta["gpt_emotion"].value_counts())

Pretrained shape:  (3000, 12, 768)   ← should be (3000, 12, 768)
Finetuned shape:   (3000, 12, 768)  ← should be (3000, 12, 768)
Metadata rows:     3000      ← should be 3000

Emotion distribution in subsample:
gpt_emotion
anger      750
disgust    750
joy        750
sadness    750
Name: count, dtype: int64
